In [1]:
import datetime
import glob
from io import StringIO
#import os
import random
import re
import time
import pickle

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.chrome.options import Options

import chromedriver_binary

import chainer
import chainer.functions as F
import chainer.links as L
from chainer import serializers

from linebot import LineBotApi
from linebot.v3.messaging import Configuration, ApiClient, MessagingApi, PushMessageRequest, TextMessage

import my_snipets
from makeRawData import makeHorseData

In [2]:
def makeRaceDateURLList(date:datetime.date) -> str:
    """
    対象日付の開催レース一覧ページのURLを生成する関数
    `date`: 対象の年月日 (datetime.dateオブジェクト)
    """
  
    #対象年月の開催カレンダーのURLを生成
    url = f"https://race.netkeiba.com/top/calendar.html?year={date.year}&month={date.month}"

    #カレンダーページのHTMLを取得
    headers = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
    html = requests.get(url, headers=headers)
    html.encoding = "EUC-JP"

    #HTMLをsoupオブジェクトに変換
    soup = BeautifulSoup(html.text)

    #開催日の一覧を取得し、開催日ページのURLを取得
    RaceCellBoxAll = soup.find_all("td",attrs={"class":"RaceCellBox"})
    
    for i in range(len(RaceCellBoxAll)):
        if RaceCellBoxAll[i].find("a") is not None:
            day = RaceCellBoxAll[i].find("span",attrs={"class":"Day"}).text
            if datetime.date(date.year, date.month, int(day)) == date:
                returnStr = (RaceCellBoxAll[i].find("a")["href"].replace("..","https://race.netkeiba.com/"))
                break

    return returnStr

In [3]:
def makeRaceURLList(tgtURL:str) -> pd.DataFrame:
    """
    開催日ページのURLから、各レースのURLリストと発走日時のデータフレーム(発走日時昇順)を作成する関数  
    `tgtURL`: 対象の開催日ページのURL (文字列)
    """
    returnDF = pd.DataFrame()
    URLList = []
    datetimeList = []
    dateStr = tgtURL.split("=")[1]

    # Seleniumを使用してブラウザを起動し、対象URLにアクセス
    option = Options()
    option.add_argument('--headless')
    browser = webdriver.Chrome(options=option)
    browser.get(tgtURL)

    # ブラウザから必要な情報を取得
    RaceListDataItemAll = browser.find_elements(By.CLASS_NAME,"RaceList_DataItem")
    for item in RaceListDataItemAll:
        spanList = item.find_element(By.CLASS_NAME,"RaceData").find_elements(By.TAG_NAME,"span")
        if len(spanList) == 4 and len(spanList[0].text) > 0:
            timeStr = spanList[0].text
        elif len(spanList) == 5 and len(spanList[1].text) > 0:
            timeStr = spanList[1].text
        else:
            timeStr = "23:59"

        URLList.append(item.find_element(By.TAG_NAME,"a").get_attribute("href"))
        datetimeList.append(datetime.datetime(int(dateStr[0:4]), int(dateStr[4:6]), int(dateStr[6:8]),int(timeStr[0:2]), int(timeStr[3:5])))

    # ブラウザを閉じる
    browser.quit()
    
    # データフレームに変換
    returnDF = pd.DataFrame({"URL": URLList,"datetime": datetimeList})
    returnDF.sort_values(by="datetime", inplace=True)
    returnDF.reset_index(drop=True, inplace=True)

    print(f"対象日付: {dateStr}" + "\n" + f"レース数: {len(returnDF)}")

    return returnDF

In [4]:
def screpingRaceCardHTML(raceURL:str):
    """
    レースカードのHTMLをスクレイピングする関数
    `RaceURL`: レースカードページのURL
    """
    raceID = re.sub(r"\D+", "", str(raceURL.split("/")[-1:]))
    option = Options()
    option.add_argument('--headless')
    browser = webdriver.Chrome(options=option)
    browser.get(raceURL)
    html = browser.page_source
    with open(f"../data/html/raceCard/{raceID}.txt","w") as f:
            f.write(html)
    browser.quit()

    print(f"[raceID:{raceID}]出馬表のHTMLをスクレイピングしました。")

    return raceID

In [11]:
def makeRaceCardDataFrame(tgtRaceID):
    """
    レースカードのHTMLからDataFrameを作成する関数
    `raceID`: レースID
    """
    with open(f"../data/html/raceCard/{tgtRaceID}.txt", "r") as f:
        html = f.read()

    returnDF = pd.read_html(StringIO(html))[0]

    # raceCardDFのカラム名を整形
    columnList = returnDF.columns.get_level_values(1)
    returnDF.columns = columnList
    
    # HTMLをsoupオブジェクトに変換
    soup = BeautifulSoup(html)

    # raceCardDFにraceID/horseID/jockeyIDを追加
    horseIDList = []
    jockeyIDList = []
    HorseListAll = soup.find_all("tr",attrs={"class":"HorseList"})
    cnt = 0
    while cnt < len(returnDF):
        horseID = HorseListAll[cnt].find("span",attrs={"class":"HorseName"}).find("a")["href"].split("/")[-1]
        if HorseListAll[cnt].find("td",attrs={"class":"Jockey"}).find("a") != None:
            jockeyID = HorseListAll[cnt].find("td",attrs={"class":"Jockey"}).find("a")["href"].split("/")[-2]
        else:
            jockeyID = "00000"
        horseIDList.append(horseID)
        jockeyIDList.append(jockeyID)
        cnt += 1

    returnDF["raceID"] = tgtRaceID
    returnDF["horseID"] = horseIDList
    returnDF["jockeyID"] = jockeyIDList

    # レース名
    raceName = soup.find("h1", attrs={"class": "RaceName"}).text.strip()

    # レース情報
    raceInfoList = soup.find("div", attrs={"class": "RaceData01"}).text.split("/")
    
    #距離
    course_len = re.findall("\d+",raceInfoList[1].strip())[0]

    #レースタイプ
    if raceInfoList[1].strip().find("障") != -1:
        raceType = "障害"
    elif raceInfoList[1].strip().find("芝") != -1:
        raceType = "芝"
    elif raceInfoList[1].strip().find("ダ") != -1:
        raceType = "ダート"

    #天候
    try:
        weather = raceInfoList[2].strip().split(":")[1]
    except IndexError:
        weather = np.nan

    #馬場状態
    try:
        groundState = raceInfoList[3].strip().split(":")[1]
        if groundState == "稍" : #出馬表では稍重は稍表記のため修正
            groundState = "稍重"
        elif groundState == "不":#出馬表では不良は不表記のため修正
            groundState = "不良"
    except IndexError:
        groundState = np.nan

    returnDF["raceName"] = raceName
    returnDF["course_len"] = course_len
    returnDF["weather"] = weather
    returnDF["raceType"] = raceType
    returnDF["groundState"] = groundState
    returnDF["date"] =  pd.to_datetime(f"{tgtyear}年{tgtmonth}月{tgtday}日",format="%Y年%m月%d日")

    return returnDF

In [12]:
class preprocessing:
    """
    出馬表のデータの前処理を実施し、スコアリング用データを作成するクラス
    """
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def raceCardPreprocess(self) -> None:
        """
        出馬表データに対して、前処理を実行するメソッド
        """
        # カラム名の整形
        newColumns = []
        for col in self.df.columns:
            newColumns.append(col.replace(" ",""))
        self.df.columns = newColumns

        #性齢 → Sex/Age
        self.df["Sex"] = self.df["性齢"].astype(str).str[:1]
        self.df["Age"] = self.df["性齢"].str.extract(r'(\d+)').astype(int)

        #馬体重(増減) → Weight/WeightVariation
        try:
            self.df["Weight"] = self.df["馬体重(増減)"].str.split("(",expand=True)[0].astype(int)
            self.df["WeightVariation"] = self.df["馬体重(増減)"].str.split("(",expand=True)[1].str[:-1].astype(int)
        except AttributeError:
            self.df["Weight"] = np.nan
            self.df["WeightVariation"] = np.nan

        #カラムの変数型の成型
        self.df["course_len"] = self.df["course_len"].astype(int)

        return

    def makedataHorseResults(self) -> None:
        """
        出馬表に存在する全ての馬の過去成績のデータを縦結合する
        """

        self.horseResults = pd.DataFrame()
        horseIDList = self.df["horseID"].tolist()
        cnt = 1
        for horseID in horseIDList:
            print("\r" + "(" + str(cnt) + "/" + str(len(horseIDList)) + ")",end="")

            dftmp = pd.read_pickle(f"../data/rawData/horse/{horseID}.pickle")

            indexdf = []
            for i in range(len(dftmp)):
                indexdf.append(horseID)

            dftmp.index = indexdf
            self.horseResults = pd.concat([self.horseResults,dftmp])
            cnt += 1
        
        # カラム名の整形
        newColumns = []
        for col in self.horseResults.columns:
            newColumns.append(col.replace(" ",""))
        self.horseResults.columns = newColumns

        return
    
    def horseResultsPreprocess(self) -> None:
        """
        過去成績のデータを前処理する関数
        """
        self.horseResults["Rank_y"] = self.horseResults["着順"].astype(str).str.strip("()降再")
        self.horseResults = self.horseResults[self.horseResults["Rank_y"].astype(str).str.contains("\d")]
        self.horseResults["Rank_y"] = self.horseResults["Rank_y"].astype(float)
        self.horseResults["Rank_Y"] = self.horseResults["Rank_y"].astype(int)

        #変数の型を修正
        self.horseResults["date"] = pd.to_datetime(self.horseResults["日付"])

        #不要変数の削除
        self.horseResults.drop(columns=["着順","日付"], inplace=True)

        return
    
    def makeVarFromHorseResults(self) -> None:
        """
        馬の過去成績から変数を作成し、出馬表データに付与し、スコアリング用データを作成するメソッド
        作成変数===================  
        pre1Rank  :前1走の順位  
        pre2Rank  :前2走の順位  
        pre3Rank  :前3走の順位  
        pre4Rank  :前4走の順位  
        pre5Rank  :前5走の順位  
        preAllRank:過去すべての順位の平均  
        pre1Term  :前1走から今回までの期間(日)  
        pre2Term  :前2走から前1走までの期間(日)  
        pre3Term  :前3走から前2走までの期間(日)  
        pre4Term  :前4走から前3走までの期間(日)  
        pre5Term  :前5走から前4走までの期間(日)
        """

        self.forScoringDF = self.df.copy()
        #過去の順位系の変数作成===================================================================================================
        #pre1Rank,pre2Rank,pre3Rank,preAllRankMean
        #馬結果テーブルを中央の結果のみにする
        forPreRank = self.horseResults[self.horseResults['開催'].str.contains("札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉")]

        #レース結果テーブルと馬結果テーブルを結合(Rank,date,)
        forPreRank2 = self.df.merge(forPreRank[["Rank_y","date"]], left_on = "horseID", right_index = True, how = "left")

        #対象レースより過去の日付の結果のみに絞る
        forPreRank2 = forPreRank2[forPreRank2["date_x"] > forPreRank2["date_y"]]
        #レースID,horseIDをキーに連番をふる(何個前のレースかの番号)
        forPreRank2["no"] = forPreRank2.groupby(["raceID","horseID"]).cumcount()

        #過去レースの順位テーブルを作成
        for i in range(5):
            preRankTmp = pd.DataFrame()
            preRankTmp = forPreRank2[forPreRank2["no"] == i][["raceID","horseID","Rank_y"]]
            preRankTmp.rename(columns={'Rank_y': f'pre{str(i + 1)}Rank'},inplace=True)
            self.forScoringDF = self.forScoringDF.merge(preRankTmp, left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

        preAllRankMean = forPreRank2.groupby(["raceID","horseID"]).mean("Rank_y")
        preAllRankMean.rename(columns={"Rank_y":"preAllRankMean"},inplace=True)
        self.forScoringDF = self.forScoringDF.merge(preAllRankMean["preAllRankMean"],left_on=["raceID","horseID"],right_index=True,how="left")
        #過去実績のない馬に対する処理を追加する必要あり

        #過去のレース間日数系の変数作成============================================================================================
        #pre1Term,pre2Term,pre3Term
        #レース結果テーブルと馬結果テーブルを結合(Rank,date,)
        forPreTerm = self.df.merge(self.horseResults[["Rank_y","date"]], left_on = "horseID", right_index = True, how = "left")
        #対象レースより過去の日付の結果のみに絞る
        forPreTerm = forPreTerm[forPreTerm["date_x"] > forPreTerm["date_y"]]
        #レースID,horseIDをキーに連番をふる(何個前のレースかの番号)
        forPreTerm["no"] = forPreTerm.groupby(["raceID","horseID"]).cumcount()

        #過去のレース間の日数を算出
        preTermFin = pd.DataFrame()
        termColumnsList = ["raceID","horseID"]
        for i in range(5):
            preTermTmp = forPreTerm[forPreTerm["no"] == i][["raceID","horseID","date_x","date_y"]]
            preTermTmp.rename(columns={"date_y":f"pre{str(i + 1)}date"},inplace=True)

            if i == 0 :
                preTermFin = preTermTmp
                preTermFin.rename(columns={"date_x":"pre0date"},inplace=True)
            else:
                preTermFin = preTermFin.merge(preTermTmp[["raceID","horseID",f"pre{str(i + 1)}date"]],left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

            preTermFin[f"pre{str(i + 1)}Term"] = preTermFin[f"pre{str(i)}date"] - preTermFin[f"pre{str(i + 1)}date"]
            preTermFin[f"pre{str(i + 1)}Term"] = preTermFin[f"pre{str(i + 1)}Term"] / datetime.timedelta(days=1)
            termColumnsList.append(f"pre{str(i + 1)}Term")

        #dftmpにレース間日数情報を結合
        self.forScoringDF = self.forScoringDF.merge(preTermFin[termColumnsList],left_on=["raceID","horseID"],right_on=["raceID","horseID"],how="left")

        return
    
    def forScoringDFPreprocess(self) -> pd.DataFrame:
        """
        スコアリング用データの前処理をするメソッド
        """
        #カラム名の成形
        self.forScoringDF = self.forScoringDF.rename(columns={"枠":"枠番","オッズ更新":"単勝"})

        #必要なカラムだけ残す
        self.forScoringDF = self.forScoringDF[[
            '枠番',
            '馬番', 
            '斤量',
            '単勝',
            '人気',
            'course_len',
            'weather',
            'raceType',
            'groundState',
            'Sex',
            'Age',
            'Weight',
            'WeightVariation',
            'pre1Rank',
            'pre2Rank',
            'pre3Rank',
            'pre4Rank',
            'pre5Rank',
            'preAllRankMean',
            'pre1Term',
            'pre2Term',
            'pre3Term',
            'pre4Term',
            'pre5Term']]
        
        #labelEncoding
        categorical_features = ["weather","raceType","groundState","Sex"]
        for col in categorical_features:
            with open(f"../data/labelencoder/{col}_labelEncoder.pickle","rb") as f:
                LE = pickle.load(f)

            LE.classes_ = np.append(LE.classes_,np.nan)
            self.forScoringDF[col] = LE.transform(self.forScoringDF[col])
            self.forScoringDF[col] = self.forScoringDF[col].astype("category")
        
        #欠損値補完
        self.forScoringDF = self.forScoringDF.fillna({
            "枠番":-1,
            "馬番":-1,
            "Weight":-1,
            "WeightVariation":-1,
            "pre1Rank":-1,
            "pre2Rank":-1,
            "pre3Rank":-1,
            "pre4Rank":-1,
            "pre5Rank":-1,
            "preAllRankMean":-1,
            "pre1Term":-1,
            "pre2Term":-1,
            "pre3Term":-1,
            "pre4Term":-1,
            "pre5Term":-1})
        
        return self.forScoringDF

# 実行

In [13]:
  # 対象日付を指定
tgtdate = datetime.date.today()
tgtyear = tgtdate.year
tgtmonth = tgtdate.month
tgtday = tgtdate.day

#レース一覧ページのURL取得
raceListURL = makeRaceDateURLList(tgtdate)

#対象日付の全レースのURLを取得
raceALLDF = makeRaceURLList(raceListURL)

#LGBMのモデルの読み込み
with open("../models/lightGBM/LGBM_1.pickle", "rb") as f:
        modelLGBM = pickle.load(f)

#NNのモデルの読み込み
class Mychain(chainer.Chain):
    def __init__(self):
        super().__init__(
            bn = L.BatchNormalization(24),
            l1 = L.Linear(None, 24),
            l2 = L.Linear(None, 12),
            l3 = L.Linear(None, 6),
            l4 = L.Linear(None, 2),
        )

    def __call__(self,x):
        h1 = F.relu(self.l1(self.bn(x)))
        h2 = F.relu(self.l2(h1))
        h3 = F.relu(self.l3(h2))
        output = self.l4(h3)

        return output
    
model = L.Classifier(Mychain())
serializers.load_npz("../models/neuralNetwork/NN_1.npz",model)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


対象日付: 20250706
レース数: 36


In [14]:
runtimeList = []
for tgt in raceALLDF["datetime"]:
    runtime = tgt + datetime.timedelta(minutes=-3)
    runtimeList.append(runtime)

In [15]:
cnt = 0
for tgttime in runtimeList:
    returnflg = 0
    while returnflg == 0:
        now = datetime.datetime.now()
        if now.hour == tgttime.hour and now.minute == tgttime.minute :
            raceID = screpingRaceCardHTML(raceALLDF["URL"][cnt])

            raceCardDF = makeRaceCardDataFrame(raceID)

            #raceCardDFのhorseIDのデータを作成
            a = makeHorseData(raceCardDF["horseID"].tolist())
            a.getHorseHTML(skipResult=False)
            a.makeHorseRawData()

            #スコアリング用データの作成
            a = preprocessing(raceCardDF)
            a.raceCardPreprocess()
            a.makedataHorseResults()
            a.horseResultsPreprocess()
            a.makeVarFromHorseResults()
            forScoringDF = a.forScoringDFPreprocess()

            #lightGBMによるスコアリング
            socredDataLGBM = modelLGBM.predict(forScoringDF)
            raceCardDF["predLGBM"] = socredDataLGBM

            #NNによるスコアリング
            forScoring_x = forScoringDF.values
            forScoring_x = forScoring_x.astype(np.float32)
            socredDataNN = F.softmax(model.predictor(forScoring_x).data)[:,1].array
            raceCardDF["predNN"] = socredDataNN


            kaisu = raceID[6:8]
            kaisaibi = raceID[8:10]

            if raceID[4:6] == "01":
                trackName = "札幌"
            elif raceID[4:6] == "02":
                trackName = "函館"
            elif raceID[4:6] == "03":
                trackName = "福島"
            elif raceID[4:6] == "04":
                trackName = "新潟"
            elif raceID[4:6] == "05":
                trackName = "東京"
            elif raceID[4:6] == "06":
                trackName = "中山"
            elif raceID[4:6] == "07":
                trackName = "中京"
            elif raceID[4:6] == "08":
                trackName = "京都"
            elif raceID[4:6] == "09":
                trackName = "阪神"
            elif raceID[4:6] == "10":
                trackName = "小倉"

            raceNo = raceID[10:]
            raceName = raceCardDF["raceName"][0]

            raceType = raceCardDF["raceType"][0]
            course_len = raceCardDF["course_len"][0]
            weather = raceCardDF["weather"][0]
            groundState = raceCardDF["groundState"][0]

            raceCardDFLGBM=raceCardDF[["馬名","オッズ更新","predLGBM"]].sort_values(by="predLGBM", ascending=False).head(3).reset_index(drop=True)
            LGBM1st = raceCardDFLGBM["馬名"][0] + "/" + str(raceCardDFLGBM["オッズ更新"][0]) + "/" + str(raceCardDFLGBM["predLGBM"][0])
            LGBM2nd = raceCardDFLGBM["馬名"][1] + "/" + str(raceCardDFLGBM["オッズ更新"][1]) + "/" + str(raceCardDFLGBM["predLGBM"][1])
            LGBM3rd = raceCardDFLGBM["馬名"][2] + "/" + str(raceCardDFLGBM["オッズ更新"][2]) + "/" + str(raceCardDFLGBM["predLGBM"][2])

            raceCardDFNN = raceCardDF[["馬名","オッズ更新","predNN"]].sort_values(by="predNN", ascending=False).head(3).reset_index(drop=True)
            NN1st = raceCardDFNN["馬名"][0] + "/" + str(raceCardDFNN["オッズ更新"][0]) + "/" + str(raceCardDFNN["predNN"][0])
            NN2nd = raceCardDFNN["馬名"][1] + "/" + str(raceCardDFNN["オッズ更新"][1]) + "/" + str(raceCardDFNN["predNN"][1])
            NN3rd = raceCardDFNN["馬名"][2] + "/" + str(raceCardDFNN["オッズ更新"][2]) + "/" + str(raceCardDFNN["predNN"][2])


            text = f"{int(kaisu)}回 {trackName} {int(kaisaibi)}日目" + "\n" + f"{int(raceNo)}R {raceName}\n" + f"{raceType}{course_len} 天気:{weather} 馬場:{groundState}\n\n" +"【lightGBM予測結果】馬名/単勝オッズ/予測値\n" + LGBM1st  + "\n" + LGBM2nd  + "\n" + LGBM3rd + "\n\n" +"【neuralNetwork予測結果】馬名/単勝オッズ/予測値\n" + NN1st  + "\n" + NN2nd  + "\n" + NN3rd

            CHANNEL_ACCESS_TOKEN = "iFSTH9Vagd7Ozy2YcYiH+xrjx+rmQL8xDWjZtAnIaT78vhIPhDdYQ0O7a1CSpuH2VxYeifzWNAdJWEgAkSHf2VNH9woaYVm2tB9HgWCFTRmfFZtDaJmTcUbZX0whj6/uQWUp9dquBfef24HYaWzejgdB04t89/1O/w1cDnyilFU="
            USER_ID = "U176770efe9b2a6d6bae891eb95ca0f45"

            configuration = Configuration(access_token=CHANNEL_ACCESS_TOKEN)
            with ApiClient(configuration) as api_client:
                messaging_api = MessagingApi(api_client)
                message = TextMessage(text=text)
                push_message_request = PushMessageRequest(to=USER_ID,messages=[message])
                messaging_api.push_message(push_message_request)          

            returnflg = 1
            cnt += 1
        else :
            time.sleep(60)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


[raceID:202502010801]出馬表のHTMLをスクレイピングしました。
(10/10)seID:2023109034] の血統情報データを作成中(10/10)/10)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


[raceID:202510020401]出馬表のHTMLをスクレイピングしました。
(16/16)seID:2023109109] の過去戦績のrawデータを作成中(16/16)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


...

[raceID:202502010812]出馬表のHTMLをスクレイピングしました。
(10/10)seID:2020100138] の過去戦績のrawデータを作成中(10/10)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


[raceID:202510020412]出馬表のHTMLをスクレイピングしました。
(16/16)seID:2022104740] の過去戦績のrawデータを作成中(16/16)

The chromedriver version (137.0.7151.68) detected in PATH at c:\Users\cyank\miniconda3\envs\keibaAnalysis\lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (138.0.7204.97); currently, chromedriver 138.0.7204.92 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry


[raceID:202503020412]出馬表のHTMLをスクレイピングしました。
(15/15)seID:2021100833] の過去戦績のrawデータを作成中(15/15)